# "THE POUR IS RIGHT" - Week 7, Colab notebook 1 of 3

# The base model, before any training

This notebook loads `Qwen/Qwen3-4B-Base` exactly as it comes and asks it to price our 200 test wines. It mirrors the Colab half of Ed's week 7 day 2.

The score here is the "before". Ed's untrained Llama scored worse than always guessing the average, so expect something poor. What matters is how far training moves it in the next notebook.

It takes about 15 minutes and is free on Colab's T4 GPU.

## Before you start

1. **Use a GPU.** In the menu: Runtime -> Change runtime type -> T4 GPU -> Save.
2. **Add your Hugging Face token as a secret.** Click the key icon in the left sidebar, add a secret named `HF_TOKEN` with your token as the value, and switch on "Notebook access".

Then run the cells in order.

In [ ]:
# Install the libraries. The versions are pinned to the ones the training code was tested against.
# This takes a minute or two. Warnings about other packages' dependencies can be ignored.

!pip install -q --upgrade transformers==5.18.0 trl==1.14.1 peft==0.21.2 accelerate bitsandbytes datasets

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed

In [ ]:
# Log in to Hugging Face with the token saved in Colab's secrets

from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Constants

HF_USER = "Warumanu"
DATASET_NAME = f"{HF_USER}/wines_prompts_lite"
BASE_MODEL = "Qwen/Qwen3-4B-Base"

In [ ]:
# Check that the GPU is there

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Using {gpu} with {memory:.0f} GB of memory")
else:
    print("No GPU found. Go to Runtime -> Change runtime type and choose T4 GPU, then start again.")

## The data

This is the dataset pushed from `week7_day2.ipynb`: each wine is a `prompt` ending in "Price is $" and a `completion` holding the price.

In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset["train"]
val = dataset["val"]
test = dataset["test"]

print(f"{len(train):,} training wines, {len(val):,} validation wines, {len(test):,} test wines")

In [ ]:
print(test[0]["prompt"])
print("--> the real price is", test[0]["completion"])

## Load the model in 4 bits

A model is billions of numbers, normally stored at 16 or 32 bits each. At full size this one would not leave room to train on a free GPU.

**Quantization** stores each number in just 4 bits. The model loses a little precision and shrinks to about a quarter of its size. This is the "Q" in QLoRA.

- `load_in_4bit`: use 4 bits per number
- `nf4`: a 4-bit scheme designed for the way a model's numbers are spread out
- `double_quant`: compress a second time, saving a little more memory
- `compute_dtype=float16`: do the arithmetic in 16 bits, which the T4 handles well

In [ ]:
# Quantization: load the model's 4 billion numbers at 4 bits each instead of 16 or 32

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

In [ ]:
# Load the tokenizer and the base model. The first run downloads about 8 GB, which takes a few minutes

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16,
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:.1f} GB")

The memory footprint should be about 3 GB. At full 32-bit size the same model would need about 16 GB.

## Ask it for a price

The model continues the prompt. We let it write up to 8 new tokens, because Qwen writes a price one character at a time (`1`, `2`, `5`, `.`, `0`, `0`), then keep only what it added.

In [ ]:
def model_predict(item):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = base_model.generate(**inputs, max_new_tokens=8)
    prompt_length = inputs["input_ids"].shape[1]
    return tokenizer.decode(output_ids[0, prompt_length:], skip_special_tokens=True)

In [ ]:
set_seed(42)

for i in range(3):
    print(f"Model says: {model_predict(test[i])!r}   Real price: {test[i]['completion']}")

The model writes a number and then carries on with more text. The evaluation takes the first number it finds.

## Score it on the 200 test wines

In [ ]:
# The evaluation harness. This is Ed's Tester from week7/util.py, with the colour limits we set for wine on day 3
# You don't need to read this cell closely: it scores a model on the first 200 test wines and draws the two charts

import re
import math
from itertools import accumulate
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.auto import tqdm

# Tell Plotly it is running in Colab, otherwise the charts come out blank
pio.renderers.default = "colab"

GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"
COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}

DEFAULT_SIZE = 200


class Tester:
    def __init__(self, predictor, data, title=None, size=DEFAULT_SIZE):
        self.predictor = predictor
        self.data = data
        self.title = title or predictor.__name__.replace("_", " ").title()
        self.size = size
        self.titles = []
        self.guesses = []
        self.truths = []
        self.errors = []
        self.colors = []

    @staticmethod
    def post_process(value):
        if isinstance(value, str):
            value = value.replace("$", "").replace(",", "")
            match = re.search(r"[-+]?\d*\.\d+|\d+", value)
            return float(match.group()) if match else 0
        else:
            return value

    def color_for(self, error, truth):
        if error < 10 or error / truth < 0.2:
            return "green"
        elif error < 20 or error / truth < 0.4:
            return "orange"
        else:
            return "red"

    def run_datapoint(self, i):
        datapoint = self.data[i]
        value = self.predictor(datapoint)
        guess = self.post_process(value)
        truth = float(datapoint["completion"])
        error = abs(guess - truth)
        color = self.color_for(error, truth)
        pieces = datapoint["prompt"].split("Wine: ")
        title = pieces[1].split("\n")[0] if len(pieces) > 1 else pieces[0]
        title = title if len(title) <= 40 else title[:40] + "..."
        return title, guess, truth, error, color

    def chart(self, title):
        df = pd.DataFrame({"truth": self.truths, "guess": self.guesses, "title": self.titles, "error": self.errors, "color": self.colors})
        df["hover"] = [f"{t}\nGuess=${g:,.2f} Actual=${y:,.2f}" for t, g, y in zip(df["title"], df["guess"], df["truth"])]
        max_val = float(max(df["truth"].max(), df["guess"].max()))
        fig = px.scatter(
            df, x="truth", y="guess", color="color",
            color_discrete_map={"green": "green", "orange": "orange", "red": "red"},
            title=title, labels={"truth": "Actual Price", "guess": "Predicted Price"}, width=800, height=600,
        )
        for tr in fig.data:
            mask = df["color"] == tr.name
            tr.customdata = df.loc[mask, ["hover"]].to_numpy()
            tr.hovertemplate = "%{customdata[0]}<extra></extra>"
            tr.marker.update(size=6)
        fig.add_trace(go.Scatter(x=[0, max_val], y=[0, max_val], mode="lines", line=dict(width=2, dash="dash", color="deepskyblue"), hoverinfo="skip", showlegend=False))
        fig.update_xaxes(range=[0, max_val])
        fig.update_yaxes(range=[0, max_val])
        fig.update_layout(showlegend=False)
        fig.show()

    def error_trend_chart(self):
        n = len(self.errors)
        x = list(range(1, n + 1))
        running_means = [s / i for s, i in zip(accumulate(self.errors), x)]
        running_squares = list(accumulate(e * e for e in self.errors))
        running_stds = [math.sqrt(max((sq / i) - (m**2), 0)) if i > 1 else 0 for i, sq, m in zip(x, running_squares, running_means)]
        ci = [1.96 * (sd / math.sqrt(i)) if i > 1 else 0 for i, sd in zip(x, running_stds)]
        upper = [m + c for m, c in zip(running_means, ci)]
        lower = [m - c for m, c in zip(running_means, ci)]
        title = f"{self.title} Error: ${running_means[-1]:,.2f} ± ${ci[-1]:,.2f}"
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=x + x[::-1], y=upper + lower[::-1], fill="toself", fillcolor="rgba(128,128,128,0.2)", line=dict(color="rgba(255,255,255,0)"), hoverinfo="skip", showlegend=False))
        fig.add_trace(go.Scatter(x=x, y=running_means, mode="lines", line=dict(width=3, color="firebrick")))
        fig.update_layout(title=title, xaxis_title="Number of Datapoints", yaxis_title="Average Absolute Error ($)", width=800, height=300, template="plotly_white", showlegend=False)
        fig.show()

    def report(self):
        average_error = sum(self.errors) / self.size
        mse = mean_squared_error(self.truths, self.guesses)
        r2 = r2_score(self.truths, self.guesses) * 100
        title = f"{self.title} results<br><b>Error:</b> ${average_error:,.2f} <b>MSE:</b> {mse:,.0f} <b>r²:</b> {r2:.1f}%"
        self.error_trend_chart()
        self.chart(title)

    def run(self):
        for i in tqdm(range(self.size)):
            title, guess, truth, error, color = self.run_datapoint(i)
            self.titles.append(title)
            self.guesses.append(guess)
            self.truths.append(truth)
            self.errors.append(error)
            self.colors.append(color)
            print(f"{COLOR_MAP[color]}${error:.0f} ", end="")
        print(RESET)
        self.report()


def evaluate(function, data, size=DEFAULT_SIZE):
    Tester(function, data, size=size).run()

In [ ]:
def base_qwen_3_4b(item):
    return model_predict(item)

set_seed(42)
evaluate(base_qwen_3_4b, test)

## What to note down

Write down the **Error** from the chart title. It is the "before" score for the fine-tuning in notebook 2.

For reference: always guessing the average scores \$21.71, and Claude Sonnet 5 scores \$9.06.